In [ ]:
# In[1]:

import pandas as pd
import pytz
from datetime import datetime

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs into DataFrames (reuse variable names if run multiple times)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Helper to compute compact timestamp info (epoch and formatted in UTC+8)
def ts_summary(series):
    if series.empty:
        return {'min_epoch': None, 'min_ts': None, 'max_epoch': None, 'max_ts': None}
    mn = int(series.min())
    mx = int(series.max())
    mn_dt = pd.to_datetime(mn, unit='s', utc=True).tz_convert(tz)
    mx_dt = pd.to_datetime(mx, unit='s', utc=True).tz_convert(tz)
    return {
        'min_epoch': mn,
        'min_ts': mn_dt.strftime('%Y-%m-%d %H:%M:%S %Z'),
        'max_epoch': mx,
        'max_ts': mx_dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    }

# Compact top-N helper returning list of (name, count)
def top_n_counts(series, n=20):
    vc = series.value_counts().head(n)
    return list(zip(vc.index.tolist(), vc.values.tolist()))

# Gather metadata for each file
metric_meta = {
    'rows': int(metric_df.shape[0]),
    'timestamp': ts_summary(metric_df['timestamp']) if 'timestamp' in metric_df.columns else None,
    'unique_cmdb_id_count': int(metric_df['cmdb_id'].nunique()) if 'cmdb_id' in metric_df.columns else 0,
    'unique_cmdb_id_sample': metric_df['cmdb_id'].dropna().unique()[:50].tolist() if 'cmdb_id' in metric_df.columns else [],
    'top_kpi_name_counts': top_n_counts(metric_df['kpi_name'], 20) if 'kpi_name' in metric_df.columns else []
}

trace_meta = {
    'rows': int(trace_df.shape[0]),
    'timestamp': ts_summary(trace_df['timestamp']) if 'timestamp' in trace_df.columns else None,
    'unique_cmdb_id_count': int(trace_df['cmdb_id'].nunique()) if 'cmdb_id' in trace_df.columns else 0,
    'unique_cmdb_id_sample': trace_df['cmdb_id'].dropna().unique()[:50].tolist() if 'cmdb_id' in trace_df.columns else [],
    'top_trace_name_counts': top_n_counts(trace_df['trace_name'], 20) if 'trace_name' in trace_df.columns else []
}

log_meta = {
    'rows': int(log_df.shape[0]),
    'timestamp': ts_summary(log_df['timestamp']) if 'timestamp' in log_df.columns else None,
    'unique_cmdb_id_count': int(log_df['cmdb_id'].nunique()) if 'cmdb_id' in log_df.columns else 0,
    'unique_cmdb_id_sample': log_df['cmdb_id'].dropna().unique()[:50].tolist() if 'cmdb_id' in log_df.columns else [],
    'top_log_name_counts': top_n_counts(log_df['log_name'], 20) if 'log_name' in log_df.columns else []
}

# For error_logs, compute message prefix (first 80 chars) top counts
if 'message' in error_df.columns:
    error_df['message_prefix'] = error_df['message'].astype(str).str.slice(0, 80)
    error_top = error_df['message_prefix'].value_counts().head(20)
    error_top_list = list(zip(error_top.index.tolist(), error_top.values.tolist()))
else:
    error_top_list = []

error_meta = {
    'rows': int(error_df.shape[0]),
    'timestamp': ts_summary(error_df['timestamp']) if 'timestamp' in error_df.columns else None,
    'unique_cmdb_id_count': int(error_df['cmdb_id'].nunique()) if 'cmdb_id' in error_df.columns else 0,
    'unique_cmdb_id_sample': error_df['cmdb_id'].dropna().unique()[:50].tolist() if 'cmdb_id' in error_df.columns else [],
    'top_message_prefix_counts': error_top_list
}

# Compose final compact JSON-like metadata
metadata = {
    'metric.csv': metric_meta,
    'trace.csv': trace_meta,
    'log.csv': log_meta,
    'error_logs.csv': error_meta
}

metadata

```
Out[1]:
```
Summary of the loaded telemetry (straightforward):

- metric.csv
  - Rows: 153,432
  - Timestamp range (UTC+8): 2022-03-21 23:30:00 — 2022-03-21 23:59:00
  - Unique cmdb_id: 59 (sample includes adservice, frontend, frontend-0/1/2, node-1..node-6, paymentservice, productcatalogservice, recommendationservice, etc.)
  - Top KPI patterns: dominated by container.node-5 metrics — e.g., container.node-5.container_network_transmit_MB.eth0 (1050 occurrences) and many other container.node-5.* metrics (counts ~1050 and ~990 among the top 20).

- trace.csv
  - Rows: 30,856
  - Timestamp range (UTC+8): 2022-03-21 23:30:00 — 2022-03-21 23:59:00
  - Unique cmdb_id: 40 (sample includes frontend-0/1/2, various service pods like checkoutservice-*, shippingservice-*, paymentservice-*)
  - Top trace features: mainly traces from frontends — e.g., trace.from_frontend-0.duration_mean, duration_p95, error_rate, row_count (each 596), then similar series for frontend-1 and frontend-2; trace.self metrics also present (~470).

- log.csv
  - Rows: 1,764
  - Timestamp range (UTC+8): 2022-03-21 23:30:00 — 2022-03-21 23:59:00
  - Unique cmdb_id: 31 (sample includes frontend-0/1/2, many service pods, and redis-cart-0)
  - Top log names: log.error_count (882) and log.row_count (882).

- error_logs.csv
  - Rows: 612
  - Timestamp range (UTC+8): 2022-03-21 23:30:01 — 2022-03-21 23:59:59
  - Unique cmdb_id: 3 (frontend-2, frontend-1, frontend-0)
  - Top message prefix: "severity: warning, message: failed to retrieve ads" — 612 occurrences (i.e., all error entries share this same prefix and are from frontend pods).

Key takeaway: metrics show heavy representation of node-5 container metrics among top KPIs; traces/logs are dominated by frontend-related series; error logs are entirely the same warning about failing to retrieve ads and originate from frontend-0/1/2.

The original code execution output of IPython Kernel is also provided below for reference:

{'metric.csv': {'rows': 153432, 'timestamp': {'min_epoch': 1647876600, 'min_ts': '2022-03-21 23:30:00 CST', 'max_epoch': 1647878340, 'max_ts': '2022-03-21 23:59:00 CST'}, 'unique_cmdb_id_count': 59, 'unique_cmdb_id_sample': ['adservice', 'adservice-0', 'adservice-1', 'adservice-2', 'adservice2', 'adservice2-0', 'cartservice', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'emailservice2-0', 'frontend', 'frontend-0', 'frontend-1', 'frontend-2', 'frontend2-0', 'node-1', 'node-2', 'node-3', 'node-4', 'node-5', 'node-6', 'paymentservice', 'paymentservice-0', 'paymentservice-1', 'paymentservice-2', 'paymentservice2-0', 'productcatalogservice', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'productcatalogservice2-0', 'recommendationservice', 'recommendationservice-0', 'recommendationservice-1'], 'top_kpi_name_counts': [('container.node-5.container_network_transmit_MB.eth0', 1050), ('container.node-5.container_network_transmit_errors.eth0', 1050), ('container.node-5.container_network_transmit_packets.eth0', 1050), ('container.node-5.container_network_transmit_packets_dropped.eth0', 1050), ('container.node-5.container_network_receive_MB.eth0', 1050), ('container.node-5.container_network_receive_errors.eth0', 1050), ('container.node-5.container_network_receive_packets.eth0', 1050), ('container.node-5.container_network_receive_packets_dropped.eth0', 1050), ('container.node-5.container_tasks_state.sleeping', 990), ('container.node-5.container_spec_cpu_quota', 990), ('container.node-5.container_memory_failures.container.pgfault', 990), ('container.node-5.container_memory_failcnt', 990), ('container.node-5.container_last_seen', 990), ('container.node-5.container_memory_working_set_MB', 990), ('container.node-5.container_fs_write_seconds./dev/vda1', 990), ('container.node-5.container_fs_usage_MB./dev/vda1', 990), ('container.node-5.container_ulimits_soft.max_open_files', 990), ('container.node-5.container_memory_cache', 990), ('container.node-5.container_sockets', 990), ('container.node-5.container_spec_cpu_period', 990)]}, 'trace.csv': {'rows': 30856, 'timestamp': {'min_epoch': 1647876600, 'min_ts': '2022-03-21 23:30:00 CST', 'max_epoch': 1647878340, 'max_ts': '2022-03-21 23:59:00 CST'}, 'unique_cmdb_id_count': 40, 'unique_cmdb_id_sample': ['adservice-0', 'adservice-1', 'adservice-2', 'adservice2-0', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'cartservice2-0', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'checkoutservice2-0', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'currencyservice2-0', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'emailservice2-0', 'frontend-0', 'frontend-1', 'frontend-2', 'frontend2-0', 'paymentservice-1', 'paymentservice-2', 'paymentservice2-0', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'productcatalogservice2-0', 'recommendationservice-0', 'recommendationservice-1', 'recommendationservice-2', 'recommendationservice2-0', 'shippingservice-0', 'shippingservice-1', 'shippingservice-2', 'shippingservice2-0', 'paymentservice-0'], 'top_trace_name_counts': [('trace.from_frontend-0.duration_mean', 596), ('trace.from_frontend-0.duration_p95', 596), ('trace.from_frontend-0.error_rate', 596), ('trace.from_frontend-0.row_count', 596), ('trace.from_frontend-1.duration_mean', 581), ('trace.from_frontend-1.duration_p95', 581), ('trace.from_frontend-1.error_rate', 581), ('trace.from_frontend-1.row_count', 581), ('trace.from_frontend-2.duration_mean', 522), ('trace.from_frontend-2.duration_p95', 522), ('trace.from_frontend-2.error_rate', 522), ('trace.from_frontend-2.row_count', 522), ('trace.self.row_count', 470), ('trace.self.error_rate', 470), ('trace.self.duration_p95', 470), ('trace.self.duration_mean', 470), ('trace.from_checkoutservice-0.row_count', 361), ('trace.from_checkoutservice-0.error_rate', 361), ('trace.from_checkoutservice-0.duration_p95', 361), ('trace.from_checkoutservice-0.duration_mean', 361)]}, 'log.csv': {'rows': 1764, 'timestamp': {'min_epoch': 1647876600, 'min_ts': '2022-03-21 23:30:00 CST', 'max_epoch': 1647878340, 'max_ts': '2022-03-21 23:59:00 CST'}, 'unique_cmdb_id_count': 31, 'unique_cmdb_id_sample': ['adservice-0', 'adservice-1', 'adservice-2', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'checkoutservice-0', 'checkoutservice-1', 'checkoutservice-2', 'currencyservice-0', 'currencyservice-1', 'currencyservice-2', 'emailservice-0', 'emailservice-1', 'emailservice-2', 'frontend-0', 'frontend-1', 'frontend-2', 'paymentservice-1', 'paymentservice-2', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'recommendationservice-0', 'recommendationservice-1', 'recommendationservice-2', 'shippingservice-0', 'shippingservice-1', 'shippingservice-2', 'paymentservice-0', 'redis-cart-0'], 'top_log_name_counts': [('log.error_count', 882), ('log.row_count', 882)]}, 'error_logs.csv': {'rows': 612, 'timestamp': {'min_epoch': 1647876601, 'min_ts': '2022-03-21 23:30:01 CST', 'max_epoch': 1647878399, 'max_ts': '2022-03-21 23:59:59 CST'}, 'unique_cmdb_id_count': 3, 'unique_cmdb_id_sample': ['frontend-2', 'frontend-1', 'frontend-0'], 'top_message_prefix_counts': [('severity: warning, message: failed to retrieve ads', 612)]}}```
```

In [ ]:
# In[2]:

# Reuse metric_df and tz from previous session
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Top 10 cmdb_id by number of metric rows
top_cmdb_series = metric_df['cmdb_id'].value_counts().head(10)

# Build compact summary for each top cmdb_id
top_kpis_by_cmdb = {}
top_list = []
for cid, cnt in top_cmdb_series.items():
    subset = metric_df[metric_df['cmdb_id'] == cid]
    mn_epoch = int(subset['timestamp'].min())
    mx_epoch = int(subset['timestamp'].max())
    mn_ts = pd.to_datetime(mn_epoch, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    mx_ts = pd.to_datetime(mx_epoch, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    top_kpis_ser = subset['kpi_name'].value_counts().head(10)
    top_kpis = list(zip(top_kpis_ser.index.tolist(), top_kpis_ser.values.tolist()))
    top_kpis_by_cmdb[cid] = {
        'rows': int(cnt),
        'min_epoch': mn_epoch,
        'min_ts': mn_ts,
        'max_epoch': mx_epoch,
        'max_ts': mx_ts,
        'top_kpi_name_counts': top_kpis
    }
    top_list.append((cid, int(cnt)))

# Compose compact JSON-like summary
summary = {
    'top_10_cmdb_by_rows': top_list,            # list of (cmdb_id, count)
    'top_kpis_by_these_cmdb': top_kpis_by_cmdb  # dict: cmdb_id -> compact metadata + top 10 kpis
}

summary

```
Out[2]:
```
Summary (compact):

- Top 10 cmdb_id by metric row count:
  1. adservice — 10,230 rows
  2. adservice2 — 9,990 rows
  3. frontend2-0 — 5,760 rows
  4. frontend-2 — 5,640 rows
  5. frontend-1 — 5,520 rows
  6. frontend-0 — 5,520 rows
  7. checkoutservice-0 — 4,260 rows
  8. checkoutservice-2 — 4,140 rows
  9. checkoutservice-1 — 3,900 rows
  10. checkoutservice2-0 — 3,660 rows

- For each of these top 10, top KPI patterns (top 10 kpi_name with counts, condensed):
  - adservice: JVM and app RPC/HTTP metrics (e.g., runtime.jvm_threads_state.WAITING, app.grpc.*, app.http.*); top entries each appear 30 times.
  - adservice2: JVM classloading and GC metrics (runtime.java_lang_ClassLoading_*, GarbageCollector_*); top entries each ~30.
  - frontend2-0: container metrics tied to node-6 (container.node-6.container_cpu_*, file_descriptors, fs_inodes); top entries each 30.
  - frontend-2: container metrics tied to node-5 (container.node-5.container_cpu_*, file_descriptors, fs_inodes); top entries each 30.
  - frontend-1: same pattern as frontend-2 (node-5 container CPU/fs metrics); top entries each 30.
  - frontend-0: same pattern as frontend-2 (node-5 container CPU/fs metrics); top entries each 30.
  - checkoutservice-0: same node-5 container CPU/fs metrics; top entries each 30.
  - checkoutservice-2: same node-5 container CPU/fs metrics; top entries each 30.
  - checkoutservice-1: same node-5 container CPU/fs metrics; top entries each 30.
  - checkoutservice2-0: same node-5 container CPU/fs metrics; top entries each 30.

- Time coverage for these components: metric timestamps span 2022-03-21 23:30:00 to 2022-03-21 23:59:00 (UTC+8).

Key takeaway: adservice/adservice2 rows are dominated by JVM and app-level metrics; most frontend and checkout pods’ top reported KPIs are container-level metrics associated with node-5 (except frontend2-0 which is tied to node-6).

The original code execution output of IPython Kernel is also provided below for reference:

{'top_10_cmdb_by_rows': [('adservice', 10230), ('adservice2', 9990), ('frontend2-0', 5760), ('frontend-2', 5640), ('frontend-1', 5520), ('frontend-0', 5520), ('checkoutservice-0', 4260), ('checkoutservice-2', 4140), ('checkoutservice-1', 3900), ('checkoutservice2-0', 3660)], 'top_kpis_by_these_cmdb': {'adservice': {'rows': 10230, 'min_epoch': 1647876600, 'min_ts': '2022-03-21 23:30:00 CST', 'max_epoch': 1647878340, 'max_ts': '2022-03-21 23:59:00 CST', 'top_kpi_name_counts': [('runtime.jvm_threads_state.WAITING', 30), ('app.grpc.count', 30), ('app.grpc.mrt', 30), ('app.grpc.rr', 30), ('app.grpc.sr', 30), ('app.http.count', 30), ('app.http.mrt', 30), ('app.http.rr', 30), ('app.http.sr', 30), ('runtime.java_lang_ClassLoading_LoadedClassCount', 30)]}, 'adservice2': {'rows': 9990, 'min_epoch': 1647876600, 'min_ts': '2022-03-21 23:30:00 CST', 'max_epoch': 1647878340, 'max_ts': '2022-03-21 23:59:00 CST', 'top_kpi_name_counts': [('runtime.jvm_threads_state.WAITING', 30), ('runtime.java_lang_ClassLoading_LoadedClassCount', 30), ('runtime.java_lang_ClassLoading_TotalLoadedClassCount', 30), ('runtime.java_lang_ClassLoading_UnloadedClassCount', 30), ('runtime.java_lang_ClassLoading_Verbose', 30), ('runtime.java_lang_Compilation_CompilationTimeMonitoringSupported', 30), ('runtime.java_lang_Compilation_TotalCompilationTime', 30), ('runtime.java_lang_GarbageCollector_CollectionCount.Copy', 30), ('runtime.java_lang_GarbageCollector_CollectionCount.MarkSweepCompact', 30), ('runtime.java_lang_GarbageCollector_CollectionTime.Copy', 30)]}, 'frontend2-0': {'rows': 5760, 'min_epoch': 1647876600, 'min_ts': '2022-03-21 23:30:00 CST', 'max_epoch': 1647878340, 'max_ts': '2022-03-21 23:59:00 CST', 'top_kpi_name_counts': [('container.node-6.container_cpu_cfs_periods', 30), ('container.node-6.container_cpu_cfs_throttled_periods', 30), ('container.node-6.container_cpu_cfs_throttled_seconds', 30), ('container.node-6.container_cpu_load_average_10s', 30), ('container.node-6.container_cpu_system_seconds', 30), ('container.node-6.container_cpu_usage_seconds', 30), ('container.node-6.container_cpu_user_seconds', 30), ('container.node-6.container_file_descriptors', 30), ('container.node-6.container_fs_inodes./dev/vda1', 30), ('container.node-6.container_fs_inodes_free./dev/vda1', 30)]}, 'frontend-2': {'rows': 5640, 'min_epoch': 1647876600, 'min_ts': '2022-03-21 23:30:00 CST', 'max_epoch': 1647878340, 'max_ts': '2022-03-21 23:59:00 CST', 'top_kpi_name_counts': [('container.node-5.container_cpu_cfs_periods', 30), ('container.node-5.container_cpu_cfs_throttled_periods', 30), ('container.node-5.container_cpu_cfs_throttled_seconds', 30), ('container.node-5.container_cpu_load_average_10s', 30), ('container.node-5.container_cpu_system_seconds', 30), ('container.node-5.container_cpu_usage_seconds', 30), ('container.node-5.container_cpu_user_seconds', 30), ('container.node-5.container_file_descriptors', 30), ('container.node-5.container_fs_inodes./dev/vda1', 30), ('container.node-5.container_fs_inodes_free./dev/vda1', 30)]}, 'frontend-1': {'rows': 5520, 'min_epoch': 1647876600, 'min_ts': '2022-03-21 23:30:00 CST', 'max_epoch': 1647878340, 'max_ts': '2022-03-21 23:59:00 CST', 'top_kpi_name_counts': [('container.node-5.container_cpu_cfs_periods', 30), ('container.node-5.container_cpu_cfs_throttled_periods', 30), ('container.node-5.container_cpu_cfs_throttled_seconds', 30), ('container.node-5.container_cpu_load_average_10s', 30), ('container.node-5.container_cpu_system_seconds', 30), ('container.node-5.container_cpu_usage_seconds', 30), ('container.node-5.container_cpu_user_seconds', 30), ('container.node-5.container_file_descriptors', 30), ('container.node-5.container_fs_inodes./dev/vda1', 30), ('container.node-5.container_fs_inodes_free./dev/vda1', 30)]}, 'frontend-0': {'rows': 5520, 'min_epoch': 1647876600, 'min_ts': '2022-03-21 23:30:00 CST', 'max_epoch': 1647878340, 'max_ts': '2022-03-21 23:59:00 CST', 'top_kpi_name_counts': [('container.node-5.container_cpu_cfs_periods', 30), ('container.node-5.container_cpu_cfs_throttled_periods', 30), ('container.node-5.container_cpu_cfs_throttled_seconds', 30), ('container.node-5.container_cpu_load_average_10s', 30), ('container.node-5.container_cpu_system_seconds', 30), ('container.node-5.container_cpu_usage_seconds', 30), ('container.node-5.container_cpu_user_seconds', 30), ('container.node-5.container_file_descriptors', 30), ('container.node-5.container_fs_inodes./dev/vda1', 30), ('container.node-5.container_fs_inodes_free./dev/vda1', 30)]}, 'checkoutservice-0': {'rows': 4260, 'min_epoch': 1647876600, 'min_ts': '2022-03-21 23:30:00 CST', 'max_epoch': 1647878340, 'max_ts': '2022-03-21 23:59:00 CST', 'top_kpi_name_counts': [('container.node-5.container_cpu_cfs_periods', 30), ('container.node-5.container_cpu_cfs_throttled_periods', 30), ('container.node-5.container_cpu_cfs_throttled_seconds', 30), ('container.node-5.container_cpu_load_average_10s', 30), ('container.node-5.container_cpu_system_seconds', 30), ('container.node-5.container_cpu_usage_seconds', 30), ('container.node-5.container_cpu_user_seconds', 30), ('container.node-5.container_file_descriptors', 30), ('container.node-5.container_fs_inodes./dev/vda1', 30), ('container.node-5.container_fs_inodes_free./dev/vda1', 30)]}, 'checkoutservice-2': {'rows': 4140, 'min_epoch': 1647876600, 'min_ts': '2022-03-21 23:30:00 CST', 'max_epoch': 1647878340, 'max_ts': '2022-03-21 23:59:00 CST', 'top_kpi_name_counts': [('container.node-5.container_cpu_cfs_periods', 30), ('container.node-5.container_cpu_cfs_throttled_periods', 30), ('container.node-5.container_cpu_cfs_throttled_seconds', 30), ('container.node-5.container_cpu_load_average_10s', 30), ('container.node-5.container_cpu_system_seconds', 30), ('container.node-5.container_cpu_usage_seconds', 30), ('container.node-5.container_cpu_user_seconds', 30), ('container.node-5.container_file_descriptors', 30), ('container.node-5.container_fs_inodes./dev/vda1', 30), ('container.node-5.container_fs_inodes_free./dev/vda1', 30)]}, 'checkoutservice-1': {'rows': 3900, 'min_epoch': 1647876600, 'min_ts': '2022-03-21 23:30:00 CST', 'max_epoch': 1647878340, 'max_ts': '2022-03-21 23:59:00 CST', 'top_kpi_name_counts': [('container.node-5.container_cpu_cfs_periods', 30), ('container.node-5.container_cpu_cfs_throttled_periods', 30), ('container.node-5.container_cpu_cfs_throttled_seconds', 30), ('container.node-5.container_cpu_load_average_10s', 30), ('container.node-5.container_cpu_system_seconds', 30), ('container.node-5.container_cpu_usage_seconds', 30), ('container.node-5.container_cpu_user_seconds', 30), ('container.node-5.container_file_descriptors', 30), ('container.node-5.container_fs_inodes./dev/vda1', 30), ('container.node-5.container_fs_inodes_free./dev/vda1', 30)]}, 'checkoutservice2-0': {'rows': 3660, 'min_epoch': 1647876600, 'min_ts': '2022-03-21 23:30:00 CST', 'max_epoch': 1647878340, 'max_ts': '2022-03-21 23:59:00 CST', 'top_kpi_name_counts': [('container.node-5.container_cpu_cfs_periods', 30), ('container.node-5.container_cpu_cfs_throttled_periods', 30), ('container.node-5.container_cpu_cfs_throttled_seconds', 30), ('container.node-5.container_cpu_load_average_10s', 30), ('container.node-5.container_cpu_system_seconds', 30), ('container.node-5.container_cpu_usage_seconds', 30), ('container.node-5.container_cpu_user_seconds', 30), ('container.node-5.container_file_descriptors', 30), ('container.node-5.container_fs_inodes./dev/vda1', 30), ('container.node-5.container_fs_inodes_free./dev/vda1', 30)]}}}```
```

In [ ]:
# In[3]:

# Reuse metric_df from previous session
import numpy as np
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

cmdb_list = ['adservice','adservice2','frontend-0','frontend-1','frontend-2','frontend2-0']

def summarize_for_cmdb(df, cmdb_id):
    sub = df[df['cmdb_id'] == cmdb_id]
    if sub.empty:
        return pd.DataFrame(columns=['kpi_name','count','min_epoch','min_ts','max_epoch','max_ts','p95','p5'])
    # Ensure numeric
    sub['value'] = pd.to_numeric(sub['value'], errors='coerce')
    # Group and aggregate
    grp = sub.groupby('kpi_name')['value'].agg(
        count = 'size',
        p95 = lambda x: x.quantile(0.95),
        p5  = lambda x: x.quantile(0.05)
    ).reset_index()
    # add min/max timestamps per kpi_name using original sub
    ts = sub.groupby('kpi_name')['timestamp'].agg(min_epoch='min', max_epoch='max').reset_index()
    res = pd.merge(grp, ts, on='kpi_name')
    # format timestamps to UTC+8 strings
    res['min_epoch'] = res['min_epoch'].astype(int)
    res['max_epoch'] = res['max_epoch'].astype(int)
    res['min_ts'] = pd.to_datetime(res['min_epoch'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    res['max_ts'] = pd.to_datetime(res['max_epoch'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    # reorder & sort by count desc, limit top 10
    res = res[['kpi_name','count','min_epoch','min_ts','max_epoch','max_ts','p95','p5']].sort_values('count', ascending=False).head(10).reset_index(drop=True)
    # Cast count to int for compactness
    res['count'] = res['count'].astype(int)
    return res

summary = {}
for cid in cmdb_list:
    summary[cid] = summarize_for_cmdb(metric_df, cid)

# Display compact JSON-like structure: dict of DataFrames (each max 10 rows)
summary

```
Out[3]:
```
Summary (straightforward, UTC+8 timestamps):

1) adservice (rows per KPI = 30; time range 2022-03-21 23:30:00 — 2022-03-21 23:59:00)
- runtime.jvm_threads_state.WAITING: count=30, p95=7.0, p5=7.0
- app.grpc.count: count=30, p95=85.55, p5=66.45
- app.grpc.mrt: count=30, p95=0.002444, p5=0.002149
- app.grpc.rr: count=30, p95=100.0, p5=100.0
- app.grpc.sr: count=30, p95=75.324138, p5=70.912121
- app.http.count: count=30, p95=24.0, p5=24.0
- app.http.mrt: count=30, p95=0.009207, p5=0.008727
- app.http.rr: count=30, p95=100.0, p5=100.0
- app.http.sr: count=30, p95=100.0, p5=100.0
- runtime.java_lang_ClassLoading_LoadedClassCount: count=30, p95=5179.0, p5≈5177.1875

2) adservice2 (rows per KPI = 30; time range 2022-03-21 23:30:00 — 2022-03-21 23:59:00)
- runtime.jvm_threads_state.WAITING: count=30, p95=7.0, p5=7.0
- runtime.java_lang_ClassLoading_LoadedClassCount: count=30, p95=5171.0, p5=5170.0
- runtime.java_lang_ClassLoading_TotalLoadedClassCount: count=30, p95=5194.0, p5=5193.0
- runtime.java_lang_ClassLoading_UnloadedClassCount: count=30, p95=23.0, p5=23.0
- runtime.java_lang_ClassLoading_Verbose: count=30, p95=0.0, p5=0.0
- runtime.java_lang_Compilation_CompilationTimeMonitoringSupported: count=30, p95=1.0, p5=1.0
- runtime.java_lang_Compilation_TotalCompilationTime: count=30, p95=100565.0, p5=100447.0
- runtime.java_lang_GarbageCollector_CollectionCount.Copy: count=30, p95≈4902.7875, p5≈4811.45
- runtime.java_lang_GarbageCollector_CollectionCount.MarkSweepCompact: count=30, p95=6.0, p5=5.0
- runtime.java_lang_GarbageCollector_CollectionTime.Copy: count=30, p95≈9672.1625, p5≈9501.875

3) frontend-0 (rows per KPI = 30; time range 2022-03-21 23:30:00 — 2022-03-21 23:59:00)
- container.node-5.container_cpu_cfs_periods: count=30, p95=135.6, p5=96.95
- container.node-5.container_cpu_cfs_throttled_periods: count=30, p95=0.0, p5=0.0
- container.node-5.container_cpu_cfs_throttled_seconds: count=30, p95=0.0, p5=0.0
- container.node-5.container_cpu_load_average_10s: count=30, p95=0.0, p5=0.0
- container.node-5.container_cpu_system_seconds: count=30, p95=0.12275, p5=0.07225
- container.node-5.container_cpu_usage_seconds: count=30, p95=0.396887, p5=0.272995
- container.node-5.container_cpu_user_seconds: count=30, p95=0.21775, p5=0.12225
- container.node-5.container_file_descriptors: count=30, p95=19.0, p5=19.0
- container.node-5.container_fs_inodes./dev/vda1: count=30, p95=0.0, p5=0.0
- container.node-5.container_fs_inodes_free./dev/vda1: count=30, p95=0.0, p5=0.0

4) frontend-1 (rows per KPI = 30; time range 2022-03-21 23:30:00 — 2022-03-21 23:59:00)
- container.node-5.container_cpu_cfs_periods: count=30, p95=107.775, p5=72.4
- container.node-5.container_cpu_cfs_throttled_periods: count=30, p95=0.0, p5=0.0
- container.node-5.container_cpu_cfs_throttled_seconds: count=30, p95=0.000549, p5=0.0
- container.node-5.container_cpu_load_average_10s: count=30, p95=0.0, p5=0.0
- container.node-5.container_cpu_system_seconds: count=30, p95=0.09275, p5=0.06
- container.node-5.container_cpu_usage_seconds: count=30, p95≈0.291444, p5≈0.209039
- container.node-5.container_cpu_user_seconds: count=30, p95=0.171, p5=0.1
- container.node-5.container_file_descriptors: count=30, p95=17.0, p5=17.0
- container.node-5.container_fs_inodes./dev/vda1: count=30, p95=0.0, p5=0.0
- container.node-5.container_fs_inodes_free./dev/vda1: count=30, p95=0.0, p5=0.0

5) frontend-2 (rows per KPI = 30; time range 2022-03-21 23:30:00 — 2022-03-21 23:59:00)
- container.node-5.container_cpu_cfs_periods: count=30, p95=41.275, p5=25.175
- container.node-5.container_cpu_cfs_throttled_periods: count=30, p95=0.0, p5=0.0
- container.node-5.container_cpu_cfs_throttled_seconds: count=30, p95=0.0, p5=0.0
- container.node-5.container_cpu_load_average_10s: count=30, p95=0.0, p5=0.0
- container.node-5.container_cpu_system_seconds: count=30, p95=0.035, p5=0.015
- container.node-5.container_cpu_usage_seconds: count=30, p95≈0.103986, p5≈0.062884
- container.node-5.container_cpu_user_seconds: count=30, p95=0.065, p5=0.0245
- container.node-5.container_file_descriptors: count=30, p95=18.0, p5=16.0
- container.node-5.container_fs_inodes./dev/vda1: count=30, p95=0.0, p5=0.0
- container.node-5.container_fs_inodes_free./dev/vda1: count=30, p95=0.0, p5=0.0

6) frontend2-0 (rows per KPI = 30; time range 2022-03-21 23:30:00 — 2022-03-21 23:59:00)
- container.node-6.container_cpu_cfs_periods: count=30, p95=91.025, p5=33.175
- container.node-6.container_cpu_cfs_throttled_periods: count=30, p95=0.0, p5=0.0
- container.node-6.container_cpu_cfs_throttled_seconds: count=30, p95=0.004281, p5=0.0
- container.node-6.container_cpu_load_average_10s: count=30, p95=0.0, p5=0.0
- container.node-6.container_cpu_system_seconds: count=30, p95=0.07925, p5=0.02725
- container.node-6.container_cpu_usage_seconds: count=30, p95≈0.250509, p5≈0.083017
- container.node-6.container_cpu_user_seconds: count=30, p95=0.144, p5=0.04225
- container.node-6.container_file_descriptors: count=30, p95=17.0, p5=17.0
- container.node-6.container_fs_inodes./dev/vda1: count=30, p95=0.0, p5=0.0
- container.node-6.container_fs_inodes_free./dev/vda1: count=30, p95=0.0, p5=0.0

Notes / takeaway:
- All selected KPIs have 30 points each and cover the same 30-minute window (2022-03-21 23:30–23:59 UTC+8).
- adservice/adservice2 top KPIs are JVM/app-level metrics (high numeric ranges for classloading/GC times).
- frontend* top KPIs are container/node-level CPU and filesystem metrics tied to node-5 (frontend2-0 uses node-6).

The original code execution output of IPython Kernel is also provided below for reference:

{'adservice':                                           kpi_name  count   min_epoch                   min_ts   max_epoch                   max_ts          p95           p5
0                runtime.jvm_threads_state.WAITING     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST     7.000000     7.000000
1                                   app.grpc.count     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST    85.550000    66.450000
2                                     app.grpc.mrt     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST     0.002444     0.002149
3                                      app.grpc.rr     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST   100.000000   100.000000
4                                      app.grpc.sr     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST    75.324138    70.912121
5                                   app.http.count     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST    24.000000    24.000000
6                                     app.http.mrt     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST     0.009207     0.008727
7                                      app.http.rr     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST   100.000000   100.000000
8                                      app.http.sr     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST   100.000000   100.000000
9  runtime.java_lang_ClassLoading_LoadedClassCount     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST  5179.000000  5177.187500, 'adservice2':                                             kpi_name  count   min_epoch                   min_ts   max_epoch                   max_ts          p95          p5
0                  runtime.jvm_threads_state.WAITING     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST       7.0000       7.000
1    runtime.java_lang_ClassLoading_LoadedClassCount     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST    5171.0000    5170.000
2  runtime.java_lang_ClassLoading_TotalLoadedClas...     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST    5194.0000    5193.000
3  runtime.java_lang_ClassLoading_UnloadedClassCount     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST      23.0000      23.000
4             runtime.java_lang_ClassLoading_Verbose     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST       0.0000       0.000
5  runtime.java_lang_Compilation_CompilationTimeM...     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST       1.0000       1.000
6  runtime.java_lang_Compilation_TotalCompilation...     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST  100565.0000  100447.000
7  runtime.java_lang_GarbageCollector_CollectionC...     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST    4902.7875    4811.450
8  runtime.java_lang_GarbageCollector_CollectionC...     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST       6.0000       5.000
9  runtime.java_lang_GarbageCollector_CollectionT...     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST    9672.1625    9501.875, 'frontend-0':                                             kpi_name  count   min_epoch                   min_ts   max_epoch                   max_ts         p95         p5
0         container.node-5.container_cpu_cfs_periods     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST  135.600000  96.950000
1  container.node-5.container_cpu_cfs_throttled_p...     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST    0.000000   0.000000
2  container.node-5.container_cpu_cfs_throttled_s...     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST    0.000000   0.000000
3    container.node-5.container_cpu_load_average_10s     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST    0.000000   0.000000
4      container.node-5.container_cpu_system_seconds     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST    0.122750   0.072250
5       container.node-5.container_cpu_usage_seconds     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST    0.396887   0.272995
6        container.node-5.container_cpu_user_seconds     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST    0.217750   0.122250
7        container.node-5.container_file_descriptors     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST   19.000000  19.000000
8     container.node-5.container_fs_inodes./dev/vda1     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST    0.000000   0.000000
9  container.node-5.container_fs_inodes_free./dev...     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST    0.000000   0.000000, 'frontend-1':                                             kpi_name  count   min_epoch                   min_ts   max_epoch                   max_ts         p95         p5
0         container.node-5.container_cpu_cfs_periods     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST  107.775000  72.400000
1  container.node-5.container_cpu_cfs_throttled_p...     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST    0.000000   0.000000
2  container.node-5.container_cpu_cfs_throttled_s...     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST    0.000549   0.000000
3    container.node-5.container_cpu_load_average_10s     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST    0.000000   0.000000
4      container.node-5.container_cpu_system_seconds     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST    0.092750   0.060000
5       container.node-5.container_cpu_usage_seconds     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST    0.291444   0.209039
6        container.node-5.container_cpu_user_seconds     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST    0.171000   0.100000
7        container.node-5.container_file_descriptors     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST   17.000000  17.000000
8     container.node-5.container_fs_inodes./dev/vda1     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST    0.000000   0.000000
9  container.node-5.container_fs_inodes_free./dev...     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST    0.000000   0.000000, 'frontend-2':                                             kpi_name  count   min_epoch                   min_ts   max_epoch                   max_ts        p95         p5
0         container.node-5.container_cpu_cfs_periods     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST  41.275000  25.175000
1  container.node-5.container_cpu_cfs_throttled_p...     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST   0.000000   0.000000
2  container.node-5.container_cpu_cfs_throttled_s...     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST   0.000000   0.000000
3    container.node-5.container_cpu_load_average_10s     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST   0.000000   0.000000
4      container.node-5.container_cpu_system_seconds     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST   0.035000   0.015000
5       container.node-5.container_cpu_usage_seconds     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST   0.103986   0.062884
6        container.node-5.container_cpu_user_seconds     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST   0.065000   0.024500
7        container.node-5.container_file_descriptors     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST  18.000000  16.000000
8     container.node-5.container_fs_inodes./dev/vda1     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST   0.000000   0.000000
9  container.node-5.container_fs_inodes_free./dev...     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST   0.000000   0.000000, 'frontend2-0':                                             kpi_name  count   min_epoch                   min_ts   max_epoch                   max_ts        p95         p5
0         container.node-6.container_cpu_cfs_periods     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST  91.025000  33.175000
1  container.node-6.container_cpu_cfs_throttled_p...     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST   0.000000   0.000000
2  container.node-6.container_cpu_cfs_throttled_s...     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST   0.004281   0.000000
3    container.node-6.container_cpu_load_average_10s     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST   0.000000   0.000000
4      container.node-6.container_cpu_system_seconds     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST   0.079250   0.027250
5       container.node-6.container_cpu_usage_seconds     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST   0.250509   0.083017
6        container.node-6.container_cpu_user_seconds     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST   0.144000   0.042250
7        container.node-6.container_file_descriptors     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST  17.000000  17.000000
8     container.node-6.container_fs_inodes./dev/vda1     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST   0.000000   0.000000
9  container.node-6.container_fs_inodes_free./dev...     30  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST   0.000000   0.000000}```
```

In [ ]:
# In[4]:

# Reuse metric_df and trace_df from previous session
import numpy as np
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

components = ['adservice','adservice2','frontend-0','frontend-1','frontend-2','frontend2-0']
incident_start = 1647876600
incident_end = 1647878340

# Classification keyword sets (all checks use lowercase)
low_keywords = ['sr','rr','success','error_rate','row_count','count']
high_keywords = ['mrt','duration','latency','p95','mean','duration_mean','duration_p95']
resource_keywords = ['cpu','memory','disk','io','error','retransmit','packets_dropped','packets','transmit','receive','throttled']

def classify_direction(name):
    n = name.lower()
    # low-priority keywords
    for kw in low_keywords:
        if kw in n:
            return 'low'
    # explicit high keywords
    for kw in high_keywords:
        if kw in n:
            return 'high'
    # resource-default -> high
    for kw in resource_keywords:
        if kw in n:
            return 'high'
    # fallback: treat as high by default (conservative)
    return 'high'

def analyze_series(df, id_col, name_col, value_col, components):
    # Ensure numeric values
    df[value_col] = pd.to_numeric(df[value_col], errors='coerce')
    # Keep only components of interest
    df_sub = df[df[id_col].isin(components)].copy()
    if df_sub.empty:
        return pd.DataFrame(columns=['component','metric_name','direction','anomaly_count','earliest_anomaly_timestamp','max_deviation_ratio','source'])
    # Compute global thresholds (p95, p5) per (component, name)
    grp = df_sub.groupby([id_col, name_col])[value_col].agg([
        ('global_count','size'),
        ('p95', lambda x: x.quantile(0.95)),
        ('p5',  lambda x: x.quantile(0.05))
    ]).reset_index()
    # Prepare to accumulate anomaly summaries
    rows = []
    # For efficiency, create in-window df
    window_mask = (df_sub['timestamp'] >= incident_start) & (df_sub['timestamp'] <= incident_end)
    df_window = df_sub[window_mask].copy()
    # Iterate groups in grp
    for _, r in grp.iterrows():
        comp = r[id_col]
        name = r[name_col]
        p95 = r['p95']
        p5 = r['p5']
        direction = classify_direction(name)
        # select window points for this group
        win_points = df_window[(df_window[id_col] == comp) & (df_window[name_col] == name)]
        if win_points.empty:
            continue
        vals = win_points[value_col].values
        ts = win_points['timestamp'].values
        # Determine anomalies based on direction
        if direction == 'high':
            # anomaly if value >= p95
            # handle NaN p95 (if all nan)
            if pd.isna(p95):
                continue
            mask = vals >= p95
            thresh = p95
            # deviation ratio for high: (value - threshold)/threshold
            if thresh == 0:
                # avoid division by zero: treat any val>0 as very large deviation
                ratios = np.where(mask, np.where(thresh==0, np.inf, (vals - thresh)/thresh), 0.0)
            else:
                ratios = np.where(mask, (vals - thresh) / thresh, 0.0)
        else:  # low
            if pd.isna(p5):
                continue
            mask = vals <= p5
            thresh = p5
            # deviation ratio for low: (threshold - value)/threshold
            if thresh == 0:
                # avoid division by zero
                ratios = np.where(mask, np.nan, 0.0)
            else:
                ratios = np.where(mask, (thresh - vals) / thresh, 0.0)
        anomaly_count = int(np.sum(mask))
        if anomaly_count == 0:
            continue
        # earliest anomaly timestamp (epoch)
        try:
            earliest_ts = int(ts[mask].min())
        except Exception:
            earliest_ts = int(ts.min())
        # max deviation ratio (ignore NaN, handle inf)
        valid_ratios = ratios[mask]
        if valid_ratios.size == 0:
            max_ratio = float(0.0)
        else:
            # replace inf with a large number for sorting; keep as is in output
            max_ratio = float(np.nanmax(valid_ratios))
            if np.isinf(max_ratio):
                max_ratio = float(np.inf)
        rows.append({
            'component': comp,
            'metric_name': name,
            'direction': direction,
            'anomaly_count': anomaly_count,
            'earliest_anomaly_timestamp': earliest_ts,
            'max_deviation_ratio': max_ratio,
            'source': 'metric' if id_col=='cmdb_id' and name_col=='kpi_name' else 'trace'
        })
    if not rows:
        return pd.DataFrame(columns=['component','metric_name','direction','anomaly_count','earliest_anomaly_timestamp','max_deviation_ratio','source'])
    out = pd.DataFrame(rows)
    return out

# Analyze metric.csv (metric_df has columns: timestamp,cmdb_id,kpi_name,value)
metric_anoms = analyze_series(metric_df, 'cmdb_id', 'kpi_name', 'value', components)

# Analyze trace.csv (trace_df has columns: timestamp,cmdb_id,trace_name,value)
trace_anoms = analyze_series(trace_df, 'cmdb_id', 'trace_name', 'value', components)

# Combine results
combined = pd.concat([metric_anoms, trace_anoms], ignore_index=True, sort=False)
if combined.empty:
    anomalies_df = combined
else:
    # Replace infinite with a large numeric sentinel for sorting, but keep original inf label for clarity
    combined['_sort_ratio'] = combined['max_deviation_ratio'].replace({np.inf: 1e12})
    anomalies_df = combined.sort_values(['anomaly_count','_sort_ratio'], ascending=[False, False]).drop(columns=['_sort_ratio']).head(50)
    # Round ratio for compactness (keep inf as is)
    def fmt_ratio(x):
        if pd.isna(x):
            return None
        if np.isinf(x):
            return float('inf')
        return float(np.round(x, 6))
    anomalies_df['max_deviation_ratio'] = anomalies_df['max_deviation_ratio'].apply(fmt_ratio)

# Keep output compact: select columns and show top rows (bounded)
anomalies_df = anomalies_df[['component','metric_name','direction','anomaly_count','earliest_anomaly_timestamp','max_deviation_ratio','source']].reset_index(drop=True)

anomalies_df.head(50)

```
Out[4]:
```
Summary (straightforward):

- Results returned: top 50 anomaly rows (bounded by the requested sort/limit).
- All 50 rows are metric-level anomalies for the component adservice.
- Pattern:
  - Each flagged metric has anomaly_count = 30 (i.e., every point in the incident window 1647876600 — 1647878340 was anomalous).
  - Earliest anomaly timestamp for all rows = 1647876600 (2022-03-21 23:30:00 UTC+8).
  - Direction for all flagged metrics = "high" (classification rules picked high for these KPI names).
  - max_deviation_ratio is often infinite (inf) — this indicates the computed threshold used for ratio was zero (division-by-zero), so values exceed a zero threshold.
- Dominant metric types flagged: many adservice JVM / garbage-collector / memory related KPIs (e.g., ClassLoading, LastGcInfo fields, JVM memory pool metrics, etc.).

Key takeaway and next steps:
- The anomalies are concentrated on adservice JVM/GC/memory metrics across the full incident window (every minute in-window was anomalous).
- Investigate adservice JVM/GC and memory settings and why thresholds are zero for many KPIs (verify metric validity and units), and inspect raw values for these KPIs to understand severity.

The original code execution output of IPython Kernel is also provided below for reference:

component                                        metric_name direction  anomaly_count  earliest_anomaly_timestamp  max_deviation_ratio  source
0   adservice             runtime.java_lang_ClassLoading_Verbose      high             30                  1647876600                  inf  metric
1   adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...      high             30                  1647876600                  inf  metric
2   adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...      high             30                  1647876600                  inf  metric
3   adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...      high             30                  1647876600                  inf  metric
4   adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...      high             30                  1647876600                  inf  metric
..        ...                                                ...       ...            ...                         ...                  ...     ...
45  adservice        runtime.jvm_buffer_pool_used_buffers.mapped      high             30                  1647876600                  inf  metric
46  adservice                       runtime.jvm_classes_unloaded      high             30                  1647876600                  inf  metric
47  adservice  runtime.jvm_gc_collection_seconds.MarkSweepCom...      high             30                  1647876600                  inf  metric
48  adservice  runtime.jvm_memory_pool_MB_init.Compressed_Cla...      high             30                  1647876600                  inf  metric
49  adservice          runtime.jvm_memory_pool_MB_init.Metaspace      high             30                  1647876600                  inf  metric

[50 rows x 7 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Reuse trace_df and tz from previous session
import numpy as np
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

components = ['adservice','adservice2']
incident_start = 1647876600
incident_end = 1647878340

# Ensure numeric values
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Focused subset
tfocus = trace_df[trace_df['cmdb_id'].isin(components)].copy()

# Compute global stats (count, p95, p5) per (cmdb_id, trace_name) using full available series
grp_stats = tfocus.groupby(['cmdb_id','trace_name'])['value'].agg(
    count='size',
    global_p95=lambda x: x.quantile(0.95),
    global_p5 =lambda x: x.quantile(0.05)
).reset_index()

# Classification rules (lowercase checks)
high_err_keywords = ['error_rate','error','err']
low_count_keywords = ['row_count','count','success','ss']
high_duration_keywords = ['duration','duration_mean','duration_p95','latency','mrt','mean']

def classify_trace(name):
    n = name.lower()
    for kw in high_err_keywords:
        if kw in n:
            return 'high'
    for kw in low_count_keywords:
        if kw in n:
            return 'low'
    for kw in high_duration_keywords:
        if kw in n:
            return 'high'
    # fallback conservative
    return 'high'

# Prepare windowed points
window_mask = (tfocus['timestamp'] >= incident_start) & (tfocus['timestamp'] <= incident_end)
tf_window = tfocus[window_mask].copy()

rows = []
# iterate over groups with thresholds computed globally
for _, r in grp_stats.iterrows():
    comp = r['cmdb_id']
    tname = r['trace_name']
    p95 = r['global_p95']
    p5 = r['global_p5']
    direction = classify_trace(tname)
    # select in-window points for this group
    points = tf_window[(tf_window['cmdb_id']==comp) & (tf_window['trace_name']==tname)]
    if points.empty:
        continue
    vals = points['value'].values
    ts = points['timestamp'].values
    # determine anomaly mask and compute ratios
    if direction == 'high':
        if pd.isna(p95):
            continue
        mask = vals >= p95
        thresh = p95
        if thresh == 0:
            # any positive value above zero considered infinite deviation
            ratios = np.where(mask, np.where(vals!=thresh, np.inf, 0.0), 0.0)
        else:
            ratios = np.where(mask, (vals - thresh) / thresh, 0.0)
    else:  # low
        if pd.isna(p5):
            continue
        mask = vals <= p5
        thresh = p5
        if thresh == 0:
            ratios = np.where(mask, np.where(vals!=thresh, np.inf, 0.0), 0.0)
        else:
            ratios = np.where(mask, (thresh - vals) / thresh, 0.0)
    anomaly_count = int(np.sum(mask))
    if anomaly_count == 0:
        continue
    earliest_ts = int(ts[mask].min())
    valid_ratios = ratios[mask]
    if valid_ratios.size == 0:
        max_ratio = 0.0
    else:
        max_ratio = float(np.nanmax(valid_ratios))
        if np.isinf(max_ratio):
            max_ratio = float('inf')
    rows.append({
        'cmdb_id': comp,
        'trace_name': tname,
        'direction': direction,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': earliest_ts,
        'global_p95': float(np.round(p95,6)) if not pd.isna(p95) else None,
        'global_p5' : float(np.round(p5,6))  if not pd.isna(p5) else None,
        'max_deviation_ratio': (float(np.round(max_ratio,6)) if np.isfinite(max_ratio) else float('inf'))
    })

# Build result DataFrame, sort and limit to top 20
if rows:
    result_df = pd.DataFrame(rows)
    # replace any NaNs
    result_df = result_df.fillna(value={'global_p95':None,'global_p5':None})
    # sort by anomaly_count desc then max_deviation_ratio desc (treat inf as largest)
    result_df['_sort_ratio'] = result_df['max_deviation_ratio'].replace({np.inf: 1e12})
    result_df = result_df.sort_values(['anomaly_count','_sort_ratio'], ascending=[False, False]).drop(columns=['_sort_ratio']).head(20).reset_index(drop=True)
else:
    result_df = pd.DataFrame(columns=['cmdb_id','trace_name','direction','anomaly_count','earliest_anomaly_timestamp','global_p95','global_p5','max_deviation_ratio'])

# Keep output compact
result_df[['cmdb_id','trace_name','direction','anomaly_count','earliest_anomaly_timestamp','global_p95','global_p5','max_deviation_ratio']]

```
Out[5]:
```
Result: No trace anomalies detected for adservice or adservice2 in the incident window.

Explanation (straightforward):
- The analysis returned an empty result table.
- That means: for every trace_name associated with cmdb_id 'adservice' or 'adservice2', either there were no trace points in the incident window (1647876600–1647878340) or none of the in-window values met the anomaly rules (>= global P95 for "high" traces or <= global P5 for "low" traces).
- In short: under the specified detection rules and time window, no trace_name triggered an anomaly for those two components.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [cmdb_id, trace_name, direction, anomaly_count, earliest_anomaly_timestamp, global_p95, global_p5, max_deviation_ratio]
Index: []```
```

In [ ]:
# In[6]:

# Reuse metric_df and tz from previous session
import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

components = ['adservice','adservice2']
substrs = ['app.grpc', 'app.http', 'error_rate', 'row_count', 'count', 'sr', 'mrt', 'duration',
           'runtime.jvm', 'GarbageCollector', 'ClassLoading', 'memory_pool', 'memory']
# normalize substrings to lowercase for case-insensitive matching
substrs = [s.lower() for s in substrs]

incident_start = 1647876600
incident_end = 1647878340

# Filter to components of interest
df_comp = metric_df[metric_df['cmdb_id'].isin(components)].copy()

# Ensure kpi_name exists and create lowercase helper
df_comp['kpi_l'] = df_comp['kpi_name'].astype(str).str.lower()

# Filter rows where kpi_name contains any of the substrings (case-insensitive)
mask_kpi = df_comp['kpi_l'].apply(lambda s: any(sub in s for sub in substrs))
df_kpi = df_comp[mask_kpi].copy()

# Ensure numeric values
df_kpi['value'] = pd.to_numeric(df_kpi['value'], errors='coerce')

# Filter to incident window (inclusive)
window_mask = (df_kpi['timestamp'] >= incident_start) & (df_kpi['timestamp'] <= incident_end)
df_window = df_kpi[window_mask].copy()

# Group by cmdb_id and kpi_name and compute required summaries
records = []
grp = df_window.groupby(['cmdb_id','kpi_name'])
for (comp, kpi), sub in grp:
    # consider only non-null numeric values for stats
    subv = sub.dropna(subset=['value'])
    if subv.empty:
        continue
    count_in_window = int(subv.shape[0])
    # min value and its earliest timestamp if multiple
    idx_min = subv['value'].idxmin()
    min_value = float(subv.loc[idx_min, 'value'])
    min_ts = int(subv.loc[idx_min, 'timestamp'])
    # max value and its earliest timestamp if multiple
    idx_max = subv['value'].idxmax()
    max_value = float(subv.loc[idx_max, 'value'])
    max_ts = int(subv.loc[idx_max, 'timestamp'])
    # mean
    mean_value = float(subv['value'].mean())
    # earliest timestamp in window and its value
    idx_earliest = subv['timestamp'].idxmin()
    earliest_ts = int(subv.loc[idx_earliest, 'timestamp'])
    val_at_earliest = float(subv.loc[idx_earliest, 'value'])
    records.append({
        'cmdb_id': comp,
        'kpi_name': kpi,
        'count_in_window': count_in_window,
        'min_value': min_value,
        'min_value_timestamp': min_ts,
        'max_value': max_value,
        'max_value_timestamp': max_ts,
        'mean_value': mean_value,
        'earliest_timestamp_in_window': earliest_ts,
        'value_at_earliest_timestamp': val_at_earliest
    })

# Build result DataFrame
result_df = pd.DataFrame(records)

# If empty, ensure consistent columns
if result_df.empty:
    result_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','count_in_window','min_value','min_value_timestamp',
        'max_value','max_value_timestamp','mean_value','earliest_timestamp_in_window','value_at_earliest_timestamp'
    ])
else:
    # Sort by count desc then max_value desc, limit top 40
    result_df = result_df.sort_values(['count_in_window','max_value'], ascending=[False, False]).head(40).reset_index(drop=True)

# Display compact JSON-like table (DataFrame)
result_df

```
Out[6]:
```
Summary (straightforward):

- Scope and counts
  - Found 40 distinct KPI series (for cmdb_id adservice and adservice2) matching the requested substrings.
  - Each series has 30 points inside the incident window (1647876600 — 1647878340), i.e., full coverage for the 30-minute window.
  - For all returned rows the earliest in-window timestamp is 1647876600.

- Notable metric values (examples from top rows)
  - adservice2.runtime.java_lang_OperatingSystem_CommittedVirtualMemorySize: count=30, min=max=mean ≈ 4.793733e+09 (value constant at timestamp 1647876600).
  - adservice.runtime.java_lang_OperatingSystem_CommittedVirtualMemorySize: count=30, value ≈ 3.660235e+09 (constant).
  - Many adservice runtime/GarbageCollector/MemoryPool metrics (e.g., LastGcInfo, MemoryPool_CollectionUsage, JVM memory pool values) show single repeated values (min = max = mean), e.g., values ≈ 1.073742e+09 or 1.048576e+08 in multiple series at timestamp 1647876600.

- Key observation
  - Most of these KPI series are effectively constant across the incident window (min == max), indicating no intra-window variability for these metrics in the dataset provided.

- Suggested next steps
  - If you expect fluctuations, inspect raw time series values and the metric collection pipeline (sampling/aggregation) to confirm data correctness and units.
  - Focus investigation on the large committed memory and GC-related metrics in adservice/adservice2 if they are relevant to the incident (values shown above).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  count_in_window     min_value  min_value_timestamp     max_value  max_value_timestamp    mean_value  earliest_timestamp_in_window  value_at_earliest_timestamp
0   adservice2  runtime.java_lang_OperatingSystem_CommittedVir...               30  4.793733e+09           1647876600  4.793733e+09           1647876600  4.793733e+09                    1647876600                 4.793733e+09
1    adservice  runtime.java_lang_OperatingSystem_CommittedVir...               30  3.660235e+09           1647876600  3.660235e+09           1647876600  3.660235e+09                    1647876600                 3.660235e+09
2    adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...               30  1.073742e+09           1647876600  1.073742e+09           1647876600  1.073742e+09                    1647876600                 1.073742e+09
3    adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...               30  1.073742e+09           1647876600  1.073742e+09           1647876600  1.073742e+09                    1647876600                 1.073742e+09
4    adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...               30  1.073742e+09           1647876600  1.073742e+09           1647876600  1.073742e+09                    1647876600                 1.073742e+09
..         ...                                                ...              ...           ...                  ...           ...                  ...           ...                           ...                          ...
35   adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...               30  1.048576e+08           1647876600  1.048576e+08           1647876600  1.048576e+08                    1647876600                 1.048576e+08
36   adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...               30  1.048576e+08           1647876600  1.048576e+08           1647876600  1.048576e+08                    1647876600                 1.048576e+08
37   adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...               30  1.048576e+08           1647876600  1.048576e+08           1647876600  1.048576e+08                    1647876600                 1.048576e+08
38   adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...               30  1.048576e+08           1647876600  1.048576e+08           1647876600  1.048576e+08                    1647876600                 1.048576e+08
39   adservice  runtime.java_lang_MemoryPool_CollectionUsage_m...               30  1.048576e+08           1647876600  1.048576e+08           1647876600  1.048576e+08                    1647876600                 1.048576e+08

[40 rows x 10 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

# Reuse trace_df, tz, incident_start/incident_end from previous session where possible
import numpy as np
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

frontends = ['frontend-0','frontend-1','frontend-2']
incident_start = 1647876600
incident_end = 1647878340

# Ensure numeric values (reuse trace_df)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Filter for frontend components
tfocus = trace_df[trace_df['cmdb_id'].isin(frontends)].copy()

# Compute global stats (count, p95, p5) per (cmdb_id, trace_name) using full available series (rule 9)
grp_stats = tfocus.groupby(['cmdb_id','trace_name'])['value'].agg(
    count='size',
    global_p95=lambda x: x.quantile(0.95),
    global_p5 =lambda x: x.quantile(0.05)
).reset_index()

# Classification keyword sets (lowercase)
high_err_keywords = ['error_rate','error','err']
low_count_keywords = ['row_count','count','success','ss']
high_duration_keywords = ['duration','duration_mean','duration_p95','latency','mrt','mean']

def classify_trace(name):
    n = name.lower()
    for kw in high_err_keywords:
        if kw in n:
            return 'high'
    for kw in low_count_keywords:
        if kw in n:
            return 'low'
    for kw in high_duration_keywords:
        if kw in n:
            return 'high'
    return 'high'  # conservative fallback

# Prepare in-window points (inclusive)
window_mask = (tfocus['timestamp'] >= incident_start) & (tfocus['timestamp'] <= incident_end)
tf_window = tfocus[window_mask].copy()

rows = []
for _, r in grp_stats.iterrows():
    comp = r['cmdb_id']
    tname = r['trace_name']
    p95 = r['global_p95']
    p5 = r['global_p5']
    direction = classify_trace(tname)
    # select in-window points for this group
    points = tf_window[(tf_window['cmdb_id']==comp) & (tf_window['trace_name']==tname)]
    if points.empty:
        continue
    vals = points['value'].values
    ts = points['timestamp'].values
    # determine anomaly mask and compute ratios
    if direction == 'high':
        if pd.isna(p95):
            continue
        mask = vals >= p95
        thresh = p95
        if thresh == 0:
            ratios = np.where(mask, np.where(vals!=thresh, np.inf, 0.0), 0.0)
        else:
            ratios = np.where(mask, (vals - thresh) / thresh, 0.0)
    else:  # low
        if pd.isna(p5):
            continue
        mask = vals <= p5
        thresh = p5
        if thresh == 0:
            ratios = np.where(mask, np.where(vals!=thresh, np.inf, 0.0), 0.0)
        else:
            ratios = np.where(mask, (thresh - vals) / thresh, 0.0)
    anomaly_count = int(np.sum(mask))
    if anomaly_count == 0:
        continue
    earliest_ts = int(ts[mask].min())
    valid_ratios = ratios[mask]
    if valid_ratios.size == 0:
        max_ratio = 0.0
    else:
        max_ratio = float(np.nanmax(valid_ratios))
        if np.isinf(max_ratio):
            max_ratio = float('inf')
    rows.append({
        'cmdb_id': comp,
        'trace_name': tname,
        'direction': direction,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': earliest_ts,
        'global_p95': (None if pd.isna(p95) else float(np.round(p95,6))),
        'global_p5':  (None if pd.isna(p5)  else float(np.round(p5,6))),
        'max_deviation_ratio': (float(np.round(max_ratio,6)) if np.isfinite(max_ratio) else float('inf'))
    })

# Build result DataFrame, sort and limit to top 20
if rows:
    result_df = pd.DataFrame(rows)
    result_df['_sort_ratio'] = result_df['max_deviation_ratio'].replace({np.inf: 1e12})
    result_df = result_df.sort_values(['anomaly_count','_sort_ratio'], ascending=[False, False]).drop(columns=['_sort_ratio']).head(20).reset_index(drop=True)
else:
    result_df = pd.DataFrame(columns=[
        'cmdb_id','trace_name','direction','anomaly_count','earliest_anomaly_timestamp','global_p95','global_p5','max_deviation_ratio'
    ])

# Compact output: selected columns only (bounded to top 20)
result_df[['cmdb_id','trace_name','direction','anomaly_count','earliest_anomaly_timestamp','global_p95','global_p5','max_deviation_ratio']]

```
Out[7]:
```
Summary (straightforward):

- Findings:
  - 20 frontend trace series flagged with anomalies (cmdb_id in frontend-0, frontend-1, frontend-2).
  - Every flagged series has anomaly_count = 30 (i.e., every point in the incident window 1647876600–1647878340 was considered anomalous).
  - Earliest anomaly for all = 1647876600 (2022-03-21 23:30:00 UTC+8).
  - All detections are classified as "high" (rules matched error/duration patterns).

- Notable detail about thresholds and severity:
  - For most flagged traces global_p95 and global_p5 are 0.0, and max_deviation_ratio = 0.0. This means the in-window values equal the threshold (zero) so ratio is zero — the detection is driven by zero thresholds rather than large positive deviations.
  - Two traces on frontend-2 (trace.to_cartservice-1.duration_mean and trace.to_cartservice-1.duration_p95) have global_p95 = 0.0 but produced max_deviation_ratio = inf, indicating non-zero values compared to a zero threshold (division-by-zero in ratio calculation).

- Implication and next steps:
  - These results indicate a data/threshold artifact: many global P95/P5 values are zero, causing all in-window points to be marked anomalous under the current rules.
  - Actionable next steps: inspect the raw trace values for these trace_names, verify how global percentiles were computed (why P95==0), and refine thresholds/rules (e.g., ignore zero thresholds or require min absolute threshold) to avoid false positives driven by zero-valued baselines.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                   trace_name direction  anomaly_count  earliest_anomaly_timestamp  global_p95  global_p5  max_deviation_ratio
0   frontend-2         trace.to_cartservice-1.duration_mean      high             30                  1647876600         0.0        0.0                  inf
1   frontend-2          trace.to_cartservice-1.duration_p95      high             30                  1647876600         0.0        0.0                  inf
2   frontend-0                   trace.from_root.error_rate      high             30                  1647876600         0.0        0.0                  0.0
3   frontend-0                        trace.self.error_rate      high             30                  1647876600         0.0        0.0                  0.0
4   frontend-0              trace.to_adservice-0.error_rate      high             30                  1647876600         0.0        0.0                  0.0
5   frontend-0              trace.to_adservice-1.error_rate      high             30                  1647876600         0.0        0.0                  0.0
6   frontend-0              trace.to_adservice-2.error_rate      high             30                  1647876600         0.0        0.0                  0.0
7   frontend-0            trace.to_cartservice-0.error_rate      high             30                  1647876600         0.0        0.0                  0.0
8   frontend-0            trace.to_cartservice-1.error_rate      high             30                  1647876600         0.0        0.0                  0.0
9   frontend-0            trace.to_cartservice-2.error_rate      high             30                  1647876600         0.0        0.0                  0.0
10  frontend-0        trace.to_currencyservice-0.error_rate      high             30                  1647876600         0.0        0.0                  0.0
11  frontend-0        trace.to_currencyservice-1.error_rate      high             30                  1647876600         0.0        0.0                  0.0
12  frontend-0        trace.to_currencyservice-2.error_rate      high             30                  1647876600         0.0        0.0                  0.0
13  frontend-0  trace.to_productcatalogservice-0.error_rate      high             30                  1647876600         0.0        0.0                  0.0
14  frontend-0  trace.to_productcatalogservice-1.error_rate      high             30                  1647876600         0.0        0.0                  0.0
15  frontend-0  trace.to_productcatalogservice-2.error_rate      high             30                  1647876600         0.0        0.0                  0.0
16  frontend-0  trace.to_recommendationservice-0.error_rate      high             30                  1647876600         0.0        0.0                  0.0
17  frontend-0  trace.to_recommendationservice-1.error_rate      high             30                  1647876600         0.0        0.0                  0.0
18  frontend-0  trace.to_recommendationservice-2.error_rate      high             30                  1647876600         0.0        0.0                  0.0
19  frontend-0        trace.to_shippingservice-0.error_rate      high             30                  1647876600         0.0        0.0                  0.0```
```

In [ ]:
# In[8]:

# Reuse previously loaded DataFrames: error_df, log_df
import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# ----- Part 1: error_logs.csv compact summary -----
# Ensure message is string and compute 80-char prefix
error_df['message'] = error_df['message'].astype(str)
error_df['message_prefix'] = error_df['message'].str.slice(0, 80)

# overall earliest timestamp row
if not error_df.empty:
    overall_earliest_epoch = int(error_df['timestamp'].min())
    overall_row = error_df[error_df['timestamp'] == overall_earliest_epoch].iloc[0]
    overall_earliest = {
        'earliest_timestamp_epoch': overall_earliest_epoch,
        'cmdb_id': overall_row['cmdb_id'],
        'message': overall_row['message']
    }
else:
    overall_earliest = {'earliest_timestamp_epoch': None, 'cmdb_id': None, 'message': None}

# per-cmdb_id aggregates
if not error_df.empty:
    grp = error_df.groupby('cmdb_id').agg(
        error_count = ('timestamp','size'),
        earliest_timestamp = ('timestamp','min'),
        latest_timestamp   = ('timestamp','max')
    ).reset_index()

    # compute top message prefix per cmdb_id
    prefix_counts = error_df.groupby(['cmdb_id','message_prefix']).size().reset_index(name='prefix_count')
    # for each cmdb_id pick top prefix by count (if tie, pick the first by descending count)
    top_prefix = prefix_counts.sort_values(['cmdb_id','prefix_count'], ascending=[True, False]).groupby('cmdb_id').first().reset_index()
    top_prefix = top_prefix.rename(columns={'message_prefix':'top_message_prefix','prefix_count':'top_message_prefix_count'})

    error_summary_df = grp.merge(top_prefix, on='cmdb_id', how='left')
    # keep columns compact and ensure integer types for counts/timestamps
    error_summary_df['error_count'] = error_summary_df['error_count'].astype(int)
    error_summary_df['earliest_timestamp'] = error_summary_df['earliest_timestamp'].astype(int)
    error_summary_df['latest_timestamp'] = error_summary_df['latest_timestamp'].astype(int)
else:
    error_summary_df = pd.DataFrame(columns=['cmdb_id','error_count','earliest_timestamp','latest_timestamp','top_message_prefix','top_message_prefix_count'])

# ----- Part 2: log.csv for frontend-0/1/2 over incident window -----
frontends = ['frontend-0','frontend-1','frontend-2']
incident_start = 1647876600
incident_end   = 1647878340

# Ensure numeric values
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# Filter to relevant cmdb_id and incident window (inclusive)
log_sub = log_df[log_df['cmdb_id'].isin(frontends)].copy()
window_mask = (log_sub['timestamp'] >= incident_start) & (log_sub['timestamp'] <= incident_end)
log_window = log_sub[window_mask].copy()

# If no rows, create empty result with expected columns
if log_window.empty:
    log_summary_df = pd.DataFrame(columns=[
        'cmdb_id','log_name','count_in_window','earliest_timestamp_in_window',
        'value_at_earliest_timestamp','mean_value_in_window','max_value_in_window'
    ])
else:
    # group and compute count, earliest timestamp, mean, max
    grp = log_window.groupby(['cmdb_id','log_name'])
    agg_basic = grp.agg(
        count_in_window = ('timestamp','size'),
        earliest_timestamp_in_window = ('timestamp','min'),
        mean_value_in_window = ('value','mean'),
        max_value_in_window = ('value','max')
    ).reset_index()

    # compute value_at_earliest_timestamp by locating the row with min timestamp per group
    # get index of earliest per group
    idx_earliest = log_window.groupby(['cmdb_id','log_name'])['timestamp'].idxmin().rename('idx_earliest').reset_index()
    # merge to get the value
    idx_earliest = idx_earliest.merge(log_window[['timestamp','value']], left_on='idx_earliest', right_index=True, how='left')
    # Note: above merge brings timestamp/value of that idx; but easier: map idx -> value directly
    earliest_values = log_window.loc[idx_earliest['idx_earliest'], ['value']].reset_index(drop=True)
    idx_earliest['value_at_earliest_timestamp'] = earliest_values['value'].values

    # combine agg_basic with earliest values
    # align by (cmdb_id,log_name)
    earliest_map = log_window.loc[log_window.groupby(['cmdb_id','log_name'])['timestamp'].idxmin(), ['cmdb_id','log_name','timestamp','value']].rename(
        columns={'timestamp':'earliest_timestamp_in_window','value':'value_at_earliest_timestamp'}
    )
    log_summary_df = agg_basic.merge(earliest_map, on=['cmdb_id','log_name','earliest_timestamp_in_window'], how='left')

    # tidy columns and types
    log_summary_df = log_summary_df[['cmdb_id','log_name','count_in_window','earliest_timestamp_in_window','value_at_earliest_timestamp','mean_value_in_window','max_value_in_window']]
    log_summary_df['count_in_window'] = log_summary_df['count_in_window'].astype(int)
    # sort by count desc and limit top 20
    log_summary_df = log_summary_df.sort_values(['count_in_window','max_value_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)

# Final outputs (compact)
error_summary_df, overall_earliest, log_summary_df

```
Out[8]:
```
Part 1 — error_logs.csv (compact)
- Overall earliest error: epoch 1647876601 → 2022-03-21 23:30:01 (UTC+8). cmdb_id: frontend-2. Message: "severity: warning, message: failed to retrieve ads".
- Per component:
  - frontend-0: 308 error rows; earliest 1647876611 → 2022-03-21 23:30:11; latest 1647878399 → 2022-03-21 23:59:59. Top message prefix (first 80 chars): "severity: warning, message: failed to retrieve..." (308 occurrences).
  - frontend-1: 230 error rows; earliest 1647876606 → 2022-03-21 23:30:06; latest 1647878398 → 2022-03-21 23:59:58. Top prefix: "severity: warning, message: failed to retrieve..." (230).
  - frontend-2: 74 error rows; earliest 1647876601 → 2022-03-21 23:30:01; latest 1647878370 → 2022-03-21 23:59:30. Top prefix: "severity: warning, message: failed to retrieve..." (74).

Part 2 — log.csv (frontends only, incident window 1647876600–1647878340)
Top rows sorted by count_in_window:
- frontend-0 / log.row_count: count=30; earliest=1647876600 → 2022-03-21 23:30:00; value_at_earliest=725.0; mean=795.966667; max=917.0
- frontend-1 / log.row_count: count=30; earliest=1647876600 → 2022-03-21 23:30:00; value_at_earliest=563.0; mean=601.9; max=752.0
- frontend-2 / log.row_count: count=30; earliest=1647876600 → 2022-03-21 23:30:00; value_at_earliest=176.0; mean=195.2; max=246.0
- frontend-0 / log.error_count: count=30; earliest=1647876600 → 2022-03-21 23:30:00; value_at_earliest=8.0; mean=10.266667; max=13.0
- frontend-1 / log.error_count: count=30; earliest=1647876600 → 2022-03-21 23:30:00; value_at_earliest=7.0; mean=7.666667; max=11.0
- frontend-2 / log.error_count: count=30; earliest=1647876600 → 2022-03-21 23:30:00; value_at_earliest=3.0; mean=2.466667; max=4.0

Key takeaways
- All error logs share the same warning prefix ("failed to retrieve ads") and originate from frontend-0/1/2, with frontend-0 having the most occurrences (308).
- Frontend log metrics (row_count and error_count) are fully covered in the incident window (30 data points each); frontend-0 shows the highest row_count and error_count values among the three.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  error_count  earliest_timestamp  latest_timestamp                                 top_message_prefix  top_message_prefix_count
0  frontend-0          308          1647876611        1647878399  severity: warning, message: failed to retrieve...                       308
1  frontend-1          230          1647876606        1647878398  severity: warning, message: failed to retrieve...                       230
2  frontend-2           74          1647876601        1647878370  severity: warning, message: failed to retrieve...                        74, {'earliest_timestamp_epoch': 1647876601, 'cmdb_id': 'frontend-2', 'message': 'severity: warning, message: failed to retrieve ads'},       cmdb_id         log_name  count_in_window  earliest_timestamp_in_window  value_at_earliest_timestamp  mean_value_in_window  max_value_in_window
0  frontend-0    log.row_count               30                    1647876600                        725.0            795.966667                917.0
1  frontend-1    log.row_count               30                    1647876600                        563.0            601.900000                752.0
2  frontend-2    log.row_count               30                    1647876600                        176.0            195.200000                246.0
3  frontend-0  log.error_count               30                    1647876600                          8.0             10.266667                 13.0
4  frontend-1  log.error_count               30                    1647876600                          7.0              7.666667                 11.0
5  frontend-2  log.error_count               30                    1647876600                          3.0              2.466667                  4.0)```
```

In [ ]:
# In[9]:

# Reuse metric_df from previous session
import numpy as np
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

components = ['adservice','adservice2']
incident_start = 1647876600
incident_end = 1647878340

# Ensure numeric values for metric_df['value']
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Rule 9: compute global thresholds BEFORE filtering to window
df_comp = metric_df[metric_df['cmdb_id'].isin(components)].copy()
thr = df_comp.groupby(['cmdb_id','kpi_name'])['value'].agg(
    global_count='size',
    global_p95=lambda x: x.quantile(0.95),
    global_p5 =lambda x: x.quantile(0.05)
).reset_index()

# Classification rules per instruction (lowercase matching)
low_keywords = ['sr','rr','count','row_count','success']
high_duration_keywords = ['mrt','duration','duration_mean','duration_p95','latency','mean','p95']
resource_keywords = ['cpu','memory','disk','io','gc','garbagecollector','garbagecollector_','garbagecollector.','heap','memory_pool']

def classify_kpi(name):
    n = name.lower()
    for kw in low_keywords:
        if kw in n:
            return 'low'
    for kw in high_duration_keywords:
        if kw in n:
            return 'high'
    for kw in resource_keywords:
        if kw in n:
            return 'high'
    # fallback conservative
    return 'high'

# Add direction and threshold choice to thr (per-series global info)
thr['direction'] = thr['kpi_name'].apply(classify_kpi)
thr['thresh_used'] = thr.apply(lambda r: r['global_p95'] if r['direction']=='high' else r['global_p5'], axis=1)

# Prepare windowed data and merge thresholds
window_df = df_comp[(df_comp['timestamp'] >= incident_start) & (df_comp['timestamp'] <= incident_end)].copy()
# If no window data, result will be empty
merged = window_df.merge(thr[['cmdb_id','kpi_name','global_p95','global_p5','direction','thresh_used']], on=['cmdb_id','kpi_name'], how='left')

# Compute per-row anomaly flags and deviation ratios
def compute_anomaly_row(row):
    val = row['value']
    dirc = row['direction']
    p95 = row['global_p95']
    p5 = row['global_p5']
    thresh = row['thresh_used']
    # default not anomaly
    is_anom = False
    ratio = np.nan
    if pd.isna(val):
        return pd.Series([False, np.nan])
    if dirc == 'high':
        if pd.isna(p95):
            is_anom = False
        else:
            is_anom = val >= p95
            if is_anom:
                if thresh == 0:
                    ratio = float('inf')
                else:
                    ratio = (val - thresh) / thresh
    else:  # low
        if pd.isna(p5):
            is_anom = False
        else:
            is_anom = val <= p5
            if is_anom:
                if thresh == 0:
                    ratio = float('inf')
                else:
                    ratio = (thresh - val) / thresh
    return pd.Series([bool(is_anom), ratio])

merged[['is_anom','ratio']] = merged.apply(compute_anomaly_row, axis=1)

# Create helper columns for grouping aggregation
merged['anom_ts'] = merged.apply(lambda r: r['timestamp'] if r['is_anom'] else np.nan, axis=1)

# Aggregate per (cmdb_id,kpi_name)
agg = merged.groupby(['cmdb_id','kpi_name','direction','thresh_used','global_p95','global_p5']).agg(
    anomaly_count = ('is_anom', 'sum'),
    earliest_anomaly_timestamp = ('anom_ts', 'min'),
    max_deviation_ratio = ('ratio', 'max'),
    points_in_window = ('value','size')
).reset_index()

# Clean types: convert earliest_anomaly_timestamp to int where notna
agg['anomaly_count'] = agg['anomaly_count'].astype(int)
agg['earliest_anomaly_timestamp'] = agg['earliest_anomaly_timestamp'].apply(lambda x: int(x) if not pd.isna(x) else None)

# Count how many KPIs examined per component (using thr which is global)
kpis_per_comp = thr.groupby('cmdb_id').size().reset_index(name='total_distinct_kpis_examined')

# For each component, build the compact summary row
summary_rows = []
for comp in components:
    total_kpis = int(kpis_per_comp.loc[kpis_per_comp['cmdb_id']==comp, 'total_distinct_kpis_examined'].iloc[0]) if comp in kpis_per_comp['cmdb_id'].values else 0
    comp_agg = agg[agg['cmdb_id']==comp].copy()
    kpis_with_anoms = int(comp_agg[comp_agg['anomaly_count']>0].shape[0])
    # kpis where threshold == 0 and anomaly_count>0
    kpis_zero_thresh_anom = int(comp_agg[(comp_agg['anomaly_count']>0) & (comp_agg['thresh_used']==0)].shape[0])
    # earliest anomaly timestamp among kpis (min over non-null earliest_anomaly_timestamp)
    earliest_ts_vals = comp_agg.loc[comp_agg['anomaly_count']>0, 'earliest_anomaly_timestamp'].dropna().astype(float)
    earliest_overall = int(earliest_ts_vals.min()) if not earliest_ts_vals.empty else None
    # top 3 kpis by non-inf max_deviation_ratio (exclude inf and nan)
    noninf = comp_agg[comp_agg['anomaly_count']>0].copy()
    noninf = noninf[noninf['max_deviation_ratio'].apply(lambda x: not (pd.isna(x) or (isinstance(x,float) and np.isinf(x))))].copy()
    # if ratio present, sort desc
    if not noninf.empty:
        noninf['max_deviation_ratio'] = noninf['max_deviation_ratio'].astype(float)
        top3 = noninf.sort_values('max_deviation_ratio', ascending=False).head(3)[['kpi_name','max_deviation_ratio']]
        # format as list of tuples for compact display
        top3_list = list(zip(top3['kpi_name'].tolist(), np.round(top3['max_deviation_ratio'].astype(float).tolist(),6)))
    else:
        top3_list = []
    summary_rows.append({
        'cmdb_id': comp,
        'total_distinct_kpis_examined': total_kpis,
        'kpis_with_anomalies': kpis_with_anoms,
        'kpis_with_zero_threshold_and_anomaly': kpis_zero_thresh_anom,
        'earliest_anomaly_timestamp_among_kpis': earliest_overall,
        'top_3_kpis_by_noninf_max_deviation': top3_list
    })

summary_df = pd.DataFrame(summary_rows)[['cmdb_id','total_distinct_kpis_examined','kpis_with_anomalies','kpis_with_zero_threshold_and_anomaly','earliest_anomaly_timestamp_among_kpis','top_3_kpis_by_noninf_max_deviation']]

# Compact output: show the 2-row summary DataFrame
summary_df

```
Out[9]:
```
Compact summary (UTC+8):

- adservice
  - total distinct KPIs examined: 341
  - KPIs with anomalies in-window: 341 (every examined KPI had >=1 anomaly point)
  - KPIs with threshold == 0 and at least one anomaly (produced 'inf' ratio): 65
  - earliest anomaly timestamp among KPIs: 1647876600 → 2022-03-21 23:30:00 CST
  - top 3 KPIs by non-inf max deviation (kpi_name, max_deviation_ratio): includes app.grpc.mrt (≈2.423644), app.http.mrt (≈0.8...), and a third KPI with smaller non-inf ratio (list truncated in output)

- adservice2
  - total distinct KPIs examined: 333
  - KPIs with anomalies in-window: 333 (every examined KPI had >=1 anomaly point)
  - KPIs with threshold == 0 and at least one anomaly: 64
  - earliest anomaly timestamp among KPIs: 1647876600 → 2022-03-21 23:30:00 CST
  - top 3 KPIs by non-inf max deviation (kpi_name, max_deviation_ratio): includes runtime.jvm_classes_loaded (≈19.0), another runtime JVM metric with large ratio, and a third KPI (list truncated in output)

Key takeaway:
- Nearly all JVM/app-level and resource KPIs for both adservice and adservice2 were flagged as anomalous during the incident window. A substantial subset (65 and 64 KPIs) used a zero threshold causing 'inf' deviation ratios — this suggests threshold artifacts (zero baselines) are driving many detections. Immediate next steps: inspect raw KPI values and why many global P95/P5 are zero, and prioritize non-inf large-deviation KPIs (e.g., app.grpc.mrt, runtime.jvm_classes_loaded) for root-cause investigation.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  total_distinct_kpis_examined  kpis_with_anomalies  kpis_with_zero_threshold_and_anomaly  earliest_anomaly_timestamp_among_kpis                 top_3_kpis_by_noninf_max_deviation
0   adservice                           341                  341                                    65                             1647876600  [(app.grpc.mrt, 2.423644), (app.http.mrt, 0.80...
1  adservice2                           333                  333                                    64                             1647876600  [(runtime.jvm_classes_loaded, 19.0), (runtime....```
```

In [ ]:
# In[10]:

# Reuse trace_df, frontends, incident_start, incident_end from previous session if available
import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure required variables exist (define if not)
try:
    frontends
except NameError:
    frontends = ['frontend-0','frontend-1','frontend-2']
try:
    incident_start, incident_end
except NameError:
    incident_start = 1647876600
    incident_end = 1647878340

# Ensure numeric values
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Filter: trace_name contains 'to_adservice' (case-sensitive) and cmdb_id in frontends and within incident window
mask_name = trace_df['trace_name'].str.contains('to_adservice', case=True, na=False)
mask_cmdb = trace_df['cmdb_id'].isin(frontends)
mask_window = (trace_df['timestamp'] >= incident_start) & (trace_df['timestamp'] <= incident_end)
df_sel = trace_df[mask_name & mask_cmdb & mask_window].copy()

# Prepare result structure
cols = ['cmdb_id','trace_name','count_in_window','earliest_timestamp_in_window',
        'value_at_earliest_timestamp','mean_value_in_window','min_value_in_window','max_value_in_window']

if df_sel.empty:
    result_df = pd.DataFrame(columns=cols)
else:
    # Group and aggregate
    agg = df_sel.groupby(['cmdb_id','trace_name']).agg(
        count_in_window = ('timestamp','size'),
        earliest_timestamp_in_window = ('timestamp','min'),
        mean_value_in_window = ('value','mean'),
        min_value_in_window = ('value','min'),
        max_value_in_window = ('value','max')
    ).reset_index()

    # Get value at earliest timestamp per group
    idx_earliest = df_sel.groupby(['cmdb_id','trace_name'])['timestamp'].idxmin()
    earliest_vals = df_sel.loc[idx_earliest, ['cmdb_id','trace_name','timestamp','value']].rename(
        columns={'timestamp':'earliest_timestamp_in_window','value':'value_at_earliest_timestamp'}
    )
    # Merge to attach value_at_earliest_timestamp
    result_df = agg.merge(earliest_vals, on=['cmdb_id','trace_name','earliest_timestamp_in_window'], how='left')

    # Reorder columns and sort as requested
    result_df = result_df[['cmdb_id','trace_name','count_in_window','earliest_timestamp_in_window',
                           'value_at_earliest_timestamp','mean_value_in_window','min_value_in_window','max_value_in_window']]
    result_df = result_df.sort_values(['count_in_window','mean_value_in_window'], ascending=[False, False]).head(50).reset_index(drop=True)

# Display compact table (up to 50 rows)
result_df

```
Out[10]:
```
Summary (straightforward):

- Scope
  - Found 36 matching frontend → to_adservice trace series (cmdb_id in frontend-0/1/2) inside the incident window.
  - Every series has full coverage in-window: count_in_window = 30 for all groups.
  - Earliest in-window timestamp for all groups = 1647876600 (2022-03-21 23:30:00 UTC+8).

- Top observations (by mean_value_in_window)
  1. frontend-0 / trace.to_adservice-1.row_count — count=30; earliest=1647876600; value_at_earliest=8.0; mean≈10.30; min=7.0; max=13.0
  2. frontend-0 / trace.to_adservice-0.row_count — count=30; earliest=1647876600; value_at_earliest=8.0; mean≈10.27; min=8.0; max=13.0
  3. frontend-0 / trace.to_adservice-2.row_count — count=30; earliest=1647876600; value_at_earliest=8.0; mean≈10.27; min=8.0; max=13.0
  4. frontend-1 / trace.to_adservice-0.row_count — count=30; earliest=1647876600; value_at_earliest=7.0; mean≈7.67; min=4.0; max=11.0
  5. frontend-1 / trace.to_adservice-1.row_count — count=30; earliest=1647876600; value_at_earliest=7.0; mean≈7.63; min=4.0; max=11.0

- Error-rate traces
  - Many to_adservice.*.error_rate series are present; all have count=30, earliest=1647876600, and mean = 0.0 (min=0.0, max=0.0) within the window.

- Key takeaways
  - Frontend→adservice row_count traffic is nonzero and highest on frontend-0 (~10 requests on average per minute to each adservice instance), with per-minute peaks up to 13.
  - Error rates toward adservice are zero in these trace metrics during the window (mean 0.0).
  - All traces are fully populated for the incident window, so analysis is based on complete per-minute series.

If you want, I can return the full compact table of all 36 rows (already computed) or highlight any specific trace for deeper inspection.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                       trace_name  count_in_window  earliest_timestamp_in_window  value_at_earliest_timestamp  mean_value_in_window  min_value_in_window  max_value_in_window
0   frontend-0   trace.to_adservice-1.row_count               30                    1647876600                          8.0             10.300000                  7.0                 13.0
1   frontend-0   trace.to_adservice-0.row_count               30                    1647876600                          8.0             10.266667                  8.0                 13.0
2   frontend-0   trace.to_adservice-2.row_count               30                    1647876600                          8.0             10.266667                  8.0                 13.0
3   frontend-1   trace.to_adservice-0.row_count               30                    1647876600                          7.0              7.666667                  4.0                 11.0
4   frontend-1   trace.to_adservice-1.row_count               30                    1647876600                          7.0              7.633333                  4.0                 11.0
..         ...                              ...              ...                           ...                          ...                   ...                  ...                  ...
31  frontend-1  trace.to_adservice-1.error_rate               30                    1647876600                          0.0              0.000000                  0.0                  0.0
32  frontend-1  trace.to_adservice-2.error_rate               30                    1647876600                          0.0              0.000000                  0.0                  0.0
33  frontend-2  trace.to_adservice-0.error_rate               30                    1647876600                          0.0              0.000000                  0.0                  0.0
34  frontend-2  trace.to_adservice-1.error_rate               30                    1647876600                          0.0              0.000000                  0.0                  0.0
35  frontend-2  trace.to_adservice-2.error_rate               30                    1647876600                          0.0              0.000000                  0.0                  0.0

[36 rows x 8 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[11]:

# Reuse metric_df and tz from previous session
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Filter cmdb_id values that contain the substring 'adservice'
mask = metric_df['cmdb_id'].astype(str).str.contains('adservice', na=False)
df_ad = metric_df[mask].copy()

# Aggregate per cmdb_id
summary = df_ad.groupby('cmdb_id').agg(
    total_rows = ('timestamp', 'size'),
    distinct_kpi_count = ('kpi_name', pd.Series.nunique),
    min_epoch = ('timestamp', 'min'),
    max_epoch = ('timestamp', 'max')
).reset_index()

# Ensure integer epochs and add UTC+8 formatted timestamps
summary['min_epoch'] = summary['min_epoch'].astype(int)
summary['max_epoch'] = summary['max_epoch'].astype(int)
summary['min_ts'] = pd.to_datetime(summary['min_epoch'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
summary['max_ts'] = pd.to_datetime(summary['max_epoch'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Reorder columns for compact display
result = summary[['cmdb_id','total_rows','distinct_kpi_count','min_epoch','min_ts','max_epoch','max_ts']].sort_values('total_rows', ascending=False).reset_index(drop=True)

result

```
Out[11]:
```
Summary (compact, UTC+8):

- adservice: 10,230 metric rows; 341 distinct KPIs; time range 2022-03-21 23:30:00 — 2022-03-21 23:59:00.
- adservice2: 9,990 metric rows; 333 distinct KPIs; time range 2022-03-21 23:30:00 — 2022-03-21 23:59:00.
- adservice2-0: 3,300 metric rows; 110 distinct KPIs; time range 2022-03-21 23:30:00 — 2022-03-21 23:59:00.
- adservice-2: 2,820 metric rows; 94 distinct KPIs; time range 2022-03-21 23:30:00 — 2022-03-21 23:59:00.
- adservice-1: 2,820 metric rows; 94 distinct KPIs; time range 2022-03-21 23:30:00 — 2022-03-21 23:59:00.
- adservice-0: 2,700 metric rows; 90 distinct KPIs; time range 2022-03-21 23:30:00 — 2022-03-21 23:59:00.

Key takeaway: The primary service-level cmdb_ids (adservice and adservice2) produce the most metric rows and the largest set of KPIs; pod-level entries (adservice-0/1/2 and adservice2-0) have fewer rows and fewer distinct KPIs. All series cover the same 30-minute window.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  total_rows  distinct_kpi_count   min_epoch                   min_ts   max_epoch                   max_ts
0     adservice       10230                 341  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST
1    adservice2        9990                 333  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST
2  adservice2-0        3300                 110  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST
3   adservice-2        2820                  94  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST
4   adservice-1        2820                  94  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST
5   adservice-0        2700                  90  1647876600  2022-03-21 23:30:00 CST  1647878340  2022-03-21 23:59:00 CST```
```

In [ ]:
# In[12]:

# Reuse metric_df and tz from previous session
import numpy as np
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

pods = ['adservice-0','adservice-1','adservice-2','adservice2-0']
incident_start = 1647876600
incident_end = 1647878340

# Ensure numeric values
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Rule 9: compute global thresholds (P95, P5) for each (cmdb_id, kpi_name) before window filtering
thr = (
    metric_df[metric_df['cmdb_id'].isin(pods)]
    .groupby(['cmdb_id','kpi_name'])['value']
    .agg(global_count='size',
         global_p95=lambda x: x.quantile(0.95),
         global_p5 =lambda x: x.quantile(0.05))
    .reset_index()
)

# Classification keywords (case-insensitive)
low_keywords = ['sr','rr','success','error_rate','row_count','count']
high_keywords = ['mrt','duration','latency','p95','mean','duration_mean','duration_p95']
resource_keywords = ['cpu','memory','disk','io','error','retransmit','packets_dropped','packets','transmit','receive','throttled','gc','garbagecollector','classloading','memory_pool']

def classify_kpi(name):
    n = str(name).lower()
    for kw in low_keywords:
        if kw in n:
            return 'low'
    for kw in high_keywords:
        if kw in n:
            return 'high'
    for kw in resource_keywords:
        if kw in n:
            return 'high'
    return 'high'

thr['direction'] = thr['kpi_name'].apply(classify_kpi)
thr['thresh_used'] = thr.apply(lambda r: r['global_p95'] if r['direction']=='high' else r['global_p5'], axis=1)

# Prepare windowed points for pods
window_df = metric_df[(metric_df['cmdb_id'].isin(pods)) & 
                      (metric_df['timestamp'] >= incident_start) & 
                      (metric_df['timestamp'] <= incident_end)].copy()

# Merge thresholds into windowed points
merged = window_df.merge(thr[['cmdb_id','kpi_name','global_p95','global_p5','direction','thresh_used']],
                         on=['cmdb_id','kpi_name'], how='left')

# Compute anomaly flag and ratio per row
def row_anomaly_ratio(val, direction, p95, p5, thresh):
    if pd.isna(val):
        return False, np.nan
    if direction == 'high':
        if pd.isna(p95):
            return False, np.nan
        is_anom = val >= p95
        if not is_anom:
            return False, np.nan
        if thresh == 0:
            return True, float('inf')
        return True, (val - thresh) / thresh
    else:  # low
        if pd.isna(p5):
            return False, np.nan
        is_anom = val <= p5
        if not is_anom:
            return False, np.nan
        if thresh == 0:
            return True, float('inf')
        return True, (thresh - val) / thresh

merged[['is_anom','ratio']] = merged.apply(
    lambda r: pd.Series(row_anomaly_ratio(r['value'], r['direction'], r['global_p95'], r['global_p5'], r['thresh_used'])),
    axis=1
)

# Aggregate per (cmdb_id, kpi_name)
grp = merged.groupby(['cmdb_id','kpi_name','thresh_used']).agg(
    anomaly_count = ('is_anom','sum'),
    earliest_anom_ts = ('timestamp', lambda x: int(x[merged.loc[x.index,'is_anom']].min()) if (merged.loc[x.index,'is_anom'].any()) else np.nan),
    max_ratio = ('ratio', 'max')
).reset_index()

# Ensure numeric types and clean earliest_anom_ts
grp['anomaly_count'] = grp['anomaly_count'].astype(int)
grp['earliest_anom_ts'] = grp['earliest_anom_ts'].apply(lambda x: int(x) if not pd.isna(x) else None)
# mark zero-threshold anomalies
grp['zero_thresh_and_anom'] = ((grp['thresh_used'] == 0) & (grp['anomaly_count'] > 0))

# Compute total distinct KPIs per pod from thr (global)
total_kpis = thr.groupby('cmdb_id')['kpi_name'].nunique().reset_index().rename(columns={'kpi_name':'total_distinct_kpis'})

# Build per-pod summary
rows = []
for pod in pods:
    total_distinct = int(total_kpis.loc[total_kpis['cmdb_id']==pod, 'total_distinct_kpis'].iloc[0]) if pod in total_kpis['cmdb_id'].values else 0
    pod_grp = grp[grp['cmdb_id']==pod].copy()
    kpis_with_anoms = int(pod_grp[pod_grp['anomaly_count']>0].shape[0])
    kpis_zero_thresh_and_anom = int(pod_grp[pod_grp['zero_thresh_and_anom']].shape[0])
    earliest_ts_vals = pod_grp.loc[pod_grp['anomaly_count']>0, 'earliest_anom_ts'].dropna().astype(float)
    earliest_overall = int(earliest_ts_vals.min()) if not earliest_ts_vals.empty else None
    # top 2 KPIs by non-inf max_deviation (exclude inf and NaN)
    noninf = pod_grp[(pod_grp['anomaly_count']>0) & (~pod_grp['max_ratio'].isin([np.inf])) & (~pod_grp['max_ratio'].isna())].copy()
    if not noninf.empty:
        noninf['max_ratio'] = noninf['max_ratio'].astype(float)
        top2 = noninf.sort_values('max_ratio', ascending=False).head(2)[['kpi_name','max_ratio']]
        top2_list = list(zip(top2['kpi_name'].tolist(), np.round(top2['max_ratio'].astype(float).tolist(),6)))
    else:
        top2_list = []
    rows.append({
        'cmdb_id': pod,
        'total_distinct_kpis': total_distinct,
        'kpis_with_anomalies_in_window': kpis_with_anoms,
        'kpis_with_zero_threshold_and_anomaly': kpis_zero_thresh_and_anom,
        'earliest_anomaly_timestamp': earliest_overall,
        'top_2_kpis_by_noninf_max_deviation': top2_list
    })

summary_df = pd.DataFrame(rows)[[
    'cmdb_id','total_distinct_kpis','kpis_with_anomalies_in_window',
    'kpis_with_zero_threshold_and_anomaly','earliest_anomaly_timestamp','top_2_kpis_by_noninf_max_deviation'
]]

# Display the compact per-pod summary (4 rows)
summary_df

```
Out[12]:
```
Compact summary (plain English, UTC+8 timestamps shown):

- adservice-0
  - total distinct KPIs examined: 90
  - KPIs with anomalies in-window: 90
  - KPIs with threshold == 0 and anomaly (inf ratio): 51
  - earliest anomaly timestamp: 1647876600 → 2022-03-21 23:30:00 CST
  - top 2 KPIs by non-inf max deviation (kpi_name, ratio) — examples: (container.node-5.container_memory_failures..., ...)

- adservice-1
  - total distinct KPIs examined: 94
  - KPIs with anomalies in-window: 94
  - KPIs with threshold == 0 and anomaly (inf ratio): 54
  - earliest anomaly timestamp: 1647876600 → 2022-03-21 23:30:00 CST
  - top 2 KPIs by non-inf max deviation — examples: (container.node-5.container_network_receive_MB..., ...)

- adservice-2
  - total distinct KPIs examined: 94
  - KPIs with anomalies in-window: 94
  - KPIs with threshold == 0 and anomaly (inf ratio): 46
  - earliest anomaly timestamp: 1647876600 → 2022-03-21 23:30:00 CST
  - top 2 KPIs by non-inf max deviation — examples: (container.node-5.container_network_receive_MB..., ...)

- adservice2-0
  - total distinct KPIs examined: 110
  - KPIs with anomalies in-window: 110
  - KPIs with threshold == 0 and anomaly (inf ratio): 62
  - earliest anomaly timestamp: 1647876600 → 2022-03-21 23:30:00 CST
  - top 2 KPIs by non-inf max deviation — examples: (container.node-6.container_network_receive_MB..., ...)

Key takeaway:
- Every examined KPI for these four adservice pods had at least one in-window anomaly under the applied rules.
- A substantial subset in each pod used a zero threshold, producing infinite deviation ratios — this indicates threshold artifacts (zero baselines) driving many detections.
- The notable non-inf deviations are primarily container-level metrics (network receive/ memory failures) tied to node-5 (for adservice-*) or node-6 (for adservice2-0). Investigate those container/node metrics and why many global percentiles are zero.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  total_distinct_kpis  kpis_with_anomalies_in_window  kpis_with_zero_threshold_and_anomaly  earliest_anomaly_timestamp                 top_2_kpis_by_noninf_max_deviation
0   adservice-0                   90                             90                                    51                  1647876600  [(container.node-5.container_memory_failures.c...
1   adservice-1                   94                             94                                    54                  1647876600  [(container.node-5.container_network_receive_M...
2   adservice-2                   94                             94                                    46                  1647876600  [(container.node-5.container_network_receive_M...
3  adservice2-0                  110                            110                                    62                  1647876600  [(container.node-6.container_network_receive_M...```
```